# 04. Structured Outputs：Grammar、Token Mask 与 Compressed FSM

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 从用户视角到实现视角

用户传入 JSON schema、regex、EBNF 或 structural tag。实现上，SGLang 要在每一步采样前知道：

> 当前 grammar 状态下，词表中哪些 token 仍然合法？

因此 structured outputs 的核心不是“生成 JSON 字符串”，而是：

1. 把约束编译成 grammar object / matcher / FSM。
2. 每步根据当前状态填充 token bitmask。
3. 在 logits 上应用 mask。
4. 采样 token 后调用 `accept_token` 推进 grammar 状态。
5. 对确定性片段尝试 jump-forward，跳过逐 token 解码。


In [ ]:
for path in [
    "python/sglang/srt/constrained/base_grammar_backend.py",
    "python/sglang/srt/constrained/grammar_manager.py",
    "python/sglang/srt/constrained/xgrammar_backend.py",
    "python/sglang/srt/constrained/outlines_backend.py",
    "python/sglang/srt/constrained/outlines_jump_forward.py",
]:
    print("\n==", path)
    for row in find_defs(path, names={"BaseGrammarObject", "BaseGrammarBackend", "GrammarManager", "XGrammarGrammar", "XGrammarGrammarBackend", "OutlinesJumpForwardMap"}):
        print(row)


## `BaseGrammarObject` 是采样循环看到的接口

无论后端是 xgrammar、llguidance 还是 outlines，scheduler/sampling 层关心的接口是一组共同方法：

- `fill_vocab_mask`
- `apply_vocab_mask`
- `accept_token`
- `rollback`
- `try_jump_forward`
- `jump_and_retokenize`

这也是新增 grammar backend 时最重要的 contract。


In [ ]:
show_lines("python/sglang/srt/constrained/base_grammar_backend.py", 30, 115)


### `BaseGrammarObject`：采样循环只依赖这组状态机方法

```python
# python/sglang/srt/constrained/base_grammar_backend.py:30-113
  30: @dataclass
  31: class GrammarStats:
  32:     compilation_time: Optional[float] = None
  33:     schema_count: Optional[int] = None
  34:     ebnf_size: Optional[int] = None
  35:     is_cache_hit: bool = False
  36:     is_grammar_aborted: bool = False
  37:     tree_traversal_time: List[float] = field(default_factory=list)
  38:     dispatch_type: Optional[str] = None
  39:     num_timeout: int = 0
  40: 
  41: 
  42: class BaseGrammarObject:
  43: 
  44:     def __init__(self):
  45:         self._finished = False
  46:         self.grammar_stats = None
  47:         self.current_token = None
  48: 
  49:     def maybe_init_reasoning(self, reasoning: bool):
  50:         pass
  51: 
  52:     def accept_token(self, token: int) -> None:
  53:         """
  54:         Accept a token in the grammar.
  55:         """
  56:         raise NotImplementedError()
  57: 
  58:     def rollback(self, k: int):
  59:         raise NotImplementedError()
  60: 
  61:     def is_terminated(self):
  62:         return False
  63: 
  64:     def allocate_vocab_mask(
  65:         self, vocab_size: int, batch_size: int, device
  66:     ) -> torch.Tensor:
  67:         raise NotImplementedError()
  68: 
  69:     def fill_vocab_mask(self, vocab_mask: torch.Tensor, idx: int) -> None:
  70:         raise NotImplementedError()
  71: 
  72:     @staticmethod
  73:     def move_vocab_mask(vocab_mask: torch.Tensor, device) -> torch.Tensor:
  74:         raise NotImplementedError()
  75: 
  76:     @staticmethod
  77:     def apply_vocab_mask(logits: torch.Tensor, vocab_mask: torch.Tensor) -> None:
  78:         raise NotImplementedError()
  79: 
  80:     def copy(self) -> "BaseGrammarObject":
  81:         return self
  82: 
  83:     @property
  84:     def finished(self):
  85:         return self._finished
  86: 
  87:     @finished.setter
  88:     def finished(self, finished):
  89:         self._finished = finished
  90: 
  91:     def try_jump_forward(self, tokenizer) -> Optional[Tuple[List[int], str]]:
  92:         """
  93:         Try to jump forward in the grammar.
  94: 
  95:         Returns:
  96:             A jump forward helper which may be used in `jump_forward_str_state`.
  97:             None if the jump forward is not possible.
  98:         """
  99:         raise NotImplementedError()
 100: 
 101:     def jump_forward_str_state(self, helper: Tuple[List[int], str]) -> Tuple[str, int]:
 102:         """
 103:         Jump forward for the grammar.
 104: 
 105:         Returns:
 106:             A tuple of the jump forward string and the next state of the grammar
 107:             (which can be used in `jump_and_retokenize` if needed).
 108:         """
 109:         raise NotImplementedError()
 110: 
 111:     def jump_and_retokenize(
 112:         self, old_output_ids: List[int], new_output_ids: List[int], next_state: int
 113:     ) -> None:
```

**读这段时抓住：**

- `accept_token` 是状态机前进；采样出 token 后必须调用它，否则下一步 mask 会停在旧状态。
- `allocate_vocab_mask` / `fill_vocab_mask` / `apply_vocab_mask` 把 grammar 状态转成 logits 过滤。
- `rollback` 是 speculative/jump-forward/retokenize 场景的保险丝，允许撤回若干 token 后重放。
- `try_jump_forward` 和 `jump_and_retokenize` 是 compressed FSM 加速的接口，不是所有 backend 都必须成功返回。


## GrammarManager：异步编译与队列

Grammar 编译可能比普通请求准备更慢。SGLang 不会让未编译完成的请求直接进入 waiting queue，而是放入 `grammar_queue`。编译结果会缓存；多 rank 下还要同步哪些请求 ready/failed。


In [ ]:
show_lines("python/sglang/srt/constrained/grammar_manager.py", 55, 145)


### `GrammarManager.process_req_with_grammar`：请求什么时候被拦在 grammar queue

```python
# python/sglang/srt/constrained/grammar_manager.py:83-134
  83:         thinking_budget = self._get_request_thinking_budget(req)
      # 注：从请求参数中取出的 strict thinking token 预算；只有 reasoning grammar 会消费它。
  84:         if thinking_budget is None:
      # 注：该请求没有 strict thinking 预算，grammar manager 不需要给 ReasonerGrammarObject 写 budget。
  85:             return
  86:         if isinstance(req.grammar, ReasonerGrammarObject):
      # 注：只有 reasoning grammar 才需要写 `max_think_tokens`；普通 JSON/regex grammar 不消费 thinking budget。
  87:             req.grammar.max_think_tokens = thinking_budget
  88: 
  89:     def process_req_with_grammar(self, req: Req) -> bool:
  90:         # Init grammar cache for this request
  91:         add_to_grammar_queue = False
      # 注：标记该请求是否因为 grammar 编译未完成而暂缓进入 scheduler waiting queue。
  92:         if (
  93:             req.sampling_params.json_schema is not None
  94:             or req.sampling_params.regex is not None
  95:             or req.sampling_params.ebnf is not None
  96:             or req.sampling_params.structural_tag is not None
  97:         ):
  98:             if self.grammar_backend is None:
      # 注：用户请求 structured output 但服务以 `--grammar-backend none` 启动，只能立即 abort 该请求。
  99:                 error_msg = "Grammar-based generation (json_schema, regex, ebnf, structural_tag) is not supported when the server is launched with --grammar-backend none"
 100:                 req.set_finish_with_abort(error_msg)
      # 注：关键调用：`req.set_finish_with_abort` - 把 grammar 编译不可用或非法 schema 转成请求级 abort，scheduler 后续会把它作为已结束请求处理。
 101:             else:
 102:                 if req.sampling_params.json_schema is not None:
      # 注：JSON schema 约束优先生成 grammar cache key，同一 schema 后续请求可复用编译结果。
 103:                     key = ("json", req.sampling_params.json_schema)
      # 注：grammar cache key 由约束类型和约束内容组成，确保 JSON/regex/EBNF/structural_tag 的编译结果互不混用。
 104:                 elif req.sampling_params.regex is not None:
      # 注：regex 约束使用 `('regex', pattern)` 作为 cache key，与 JSON/EBNF 编译结果隔离。
 105:                     key = ("regex", req.sampling_params.regex)
      # 注：grammar cache key 由约束类型和约束内容组成，确保 JSON/regex/EBNF/structural_tag 的编译结果互不混用。
 106:                 elif req.sampling_params.ebnf is not None:
      # 注：EBNF 约束使用独立 cache key，避免和 regex/JSON 的 compiled grammar 混用。
 107:                     key = ("ebnf", req.sampling_params.ebnf)
      # 注：grammar cache key 由约束类型和约束内容组成，确保 JSON/regex/EBNF/structural_tag 的编译结果互不混用。
 108:                 elif req.sampling_params.structural_tag:
      # 注：structural tag 走同一 grammar cache/Future 管线，但 key 类型与 JSON/regex/EBNF 分开。
 109:                     key = ("structural_tag", req.sampling_params.structural_tag)
      # 注：grammar cache key 由约束类型和约束内容组成，确保 JSON/regex/EBNF/structural_tag 的编译结果互不混用。
 110: 
 111:                 value, cache_hit = self.grammar_backend.get_cached_or_future_value(
      # 注：grammar backend 返回 compiled grammar 或 Future；cache_hit 决定请求能否立即进入 waiting queue。
 112:                     key, req.require_reasoning
 113:                 )
 114:                 req.grammar = value
      # 注：Req 上的 grammar 字段会被 sampling 路径读取，用于每步填充并应用 token bitmask。
 115: 
 116:                 if not cache_hit:
      # 注：grammar cache miss 返回 Future，请求必须进入 grammar_queue 等待编译完成。
 117:                     req.grammar_key = key
      # 注：保存 cache miss 时的 grammar key，Future 完成或失败后要用它回写 grammar backend cache。
 118:                     add_to_grammar_queue = True
      # 注：标记该请求是否因为 grammar 编译未完成而暂缓进入 scheduler waiting queue。
 119:                 else:
 120:                     if isinstance(
 121:                         value, InvalidGrammarObject
 122:                     ):  # We hit a cached invalid grammar.
 123:                         error_msg = (
 124:                             f"Failed to compile {key[0]} grammar: {value.error_message}"
 125:                         )
 126:                         req.set_finish_with_abort(error_msg)
      # 注：关键调用：`req.set_finish_with_abort` - 把 grammar 编译不可用或非法 schema 转成请求级 abort，scheduler 后续会把它作为已结束请求处理。
 127:                     else:
 128:                         self._apply_request_reasoning_budget(req)
      # 注：关键调用：`self._apply_request_reasoning_budget` - 把 strict thinking/reasoning budget 写入 grammar object，使 reasoning token 数受同一套 token filter 约束。
 129:         elif self._enable_strict_thinking:
      # 注：没有显式 structured output 时，strict thinking 仍会初始化 reasoning grammar 来约束思考段 token。
 130:             grammar_obj = self.grammar_backend.init_strict_reasoning_grammar(
 131:                 req.require_reasoning
 132:             )
 133:             if grammar_obj is not None:
      # 注：strict thinking backend 成功创建 grammar 后，把它挂到 Req，后续 sampling 会应用 token filter。
 134:                 req.grammar = grammar_obj
      # 注：Req 上的 grammar 字段会被 sampling 路径读取，用于每步填充并应用 token bitmask。
```

**读这段时抓住：**

- 只要 sampling params 中有 JSON schema、regex、EBNF 或 structural tag，就会构造 cache key。
- `get_cached_or_future_value` 可能返回现成 grammar，也可能返回 Future；Future 未完成时请求不能进入普通 waiting queue。
- invalid grammar 会被缓存为 `InvalidGrammarObject`，避免同一个坏 schema 反复编译。
- strict thinking 复用同一套 grammar/token filter 机制，所以这里还处理 reasoning budget。


### `GrammarManager.get_ready_grammar_requests`：多 rank 下 ready/failed 要同步

```python
# python/sglang/srt/constrained/grammar_manager.py:136-212
 136: 
 137:         if add_to_grammar_queue:
      # 注：只有 grammar 编译 Future 未完成的请求才进入 grammar_queue；cache hit 可以直接进 scheduler waiting queue。
 138:             self.grammar_queue.append(req)
      # 注：关键调用：`self.grammar_queue.append` - grammar Future 未完成时请求先留在 grammar_queue，不能进入普通 waiting queue。
 139: 
 140:         return add_to_grammar_queue
 141: 
 142:     def get_ready_grammar_requests(self) -> List[Req]:
 143:         """
 144:         Move requests whose grammar objects are ready from grammar_queue to waiting_queue.
 145: 
 146:         Rank i returns two sets ready_reqs_i, failed_reqs_i
 147:         ready_reqs_all = all_gather(ready_reqs_i)
 148:         failed_reqs_all = all_gather(failed_reqs_i)
 149: 
 150:         ready_reqs = intersect(ready_reqs_all)
 151:         failed_reqs = union(failed_reqs_all)
 152:         """
 153:         assert self.grammar_backend
 154:         ready_req_idxs: set[int] = set()
      # 注：本 rank 已完成 grammar 编译或已 abort 的 grammar_queue 索引集合。
 155:         failed_req_idxs: set[int] = set()
      # 注：本 rank 轮询超时的 grammar_queue 索引集合，后续会同步到所有 rank。
 156: 
 157:         # Poll for ready requests
 158:         start_time = time.perf_counter()
 159:         while time.perf_counter() - start_time < self.SGLANG_GRAMMAR_POLL_INTERVAL:
 160:             for i, req in enumerate(self.grammar_queue):
 161:                 if i in ready_req_idxs:
      # 注：同一轮 polling 已判定 ready 的请求不重复检查，避免重复访问 Future 状态。
 162:                     continue
 163: 
 164:                 if req.finished() or req.grammar is None:  # It is aborted by AbortReq
 165:                     ready_req_idxs.add(i)
 166:                     continue
 167: 
 168:                 assert isinstance(req.grammar, futures.Future), f"{req=}"
 169:                 if req.grammar.done():
      # 注：grammar 编译 Future 完成后，请求可以从 grammar_queue 转回普通 waiting queue。
 170:                     ready_req_idxs.add(i)
 171: 
 172:             # Sleep a bit to avoid busy waiting
 173:             time.sleep(self.SGLANG_GRAMMAR_POLL_INTERVAL / 10)
 174: 
 175:         # Check failed requests
 176:         for i, req in enumerate(self.grammar_queue):
 177:             if i not in ready_req_idxs:
 178:                 self.grammar_queue[i].grammar_wait_ct += 1
 179:                 if (
 180:                     self.grammar_queue[i].grammar_wait_ct
 181:                     >= self.SGLANG_GRAMMAR_MAX_POLL_ITERATIONS
 182:                 ):
 183:                     # Timeout after max poll iterations
 184:                     # The actual waiting time is SGLANG_GRAMMAR_MAX_POLL_ITERATIONS * max(SGLANG_GRAMMAR_POLL_INTERVAL, GPU_forward_batch_latency)
 185:                     failed_req_idxs.add(i)
 186: 
 187:         # Sync ready and failed requests across all ranks
 188:         if self.grammar_sync_size == 1:
      # 注：单 rank 不需要 all-gather，直接使用本地 ready/failed 集合。
 189:             synced_ready_req_idxs = ready_req_idxs
      # 注：多 rank 下取 ready 交集，保证所有 rank 都准备好同一批 grammar 请求后再放行。
 190:             synced_failed_req_idxs = failed_req_idxs
      # 注：多 rank 下取 failed 并集，只要任一 rank 编译失败/超时，所有 rank 都按失败处理。
 191:         else:
 192:             all_gather_output = [None] * self.grammar_sync_size
 193:             torch.distributed.all_gather_object(
      # 注：关键调用：`torch.distributed.all_gather_object` - 跨 rank 同步 grammar ready/failed 集合，保证各 rank batch 一致。
 194:                 all_gather_output,
 195:                 (ready_req_idxs, failed_req_idxs),
 196:                 group=self.grammar_sync_group,
 197:             )
 198:             synced_ready_req_idxs = set.intersection(*[x[0] for x in all_gather_output])
      # 注：多 rank 下取 ready 交集，保证所有 rank 都准备好同一批 grammar 请求后再放行。
 199:             synced_failed_req_idxs = set.union(*[x[1] for x in all_gather_output])
      # 注：多 rank 下取 failed 并集，只要任一 rank 编译失败/超时，所有 rank 都按失败处理。
 200: 
 201:         # Return ready requests
 202:         return_reqs: List[Req] = []
      # 注：从 grammar_queue 释放出来、准备重新进入 scheduler waiting queue 的请求列表。
 203:         for i in synced_ready_req_idxs:
 204:             req = self.grammar_queue[i]
 205:             return_reqs.append(req)
 206:             if req.finished() or req.grammar is None:  # It is aborted by AbortReq
 207:                 continue
 208: 
 209:             assert isinstance(req.grammar, futures.Future) and req.grammar_key
 210:             try:
 211:                 req.grammar = req.grammar.result()
      # 注：Req 上的 grammar 字段会被 sampling 路径读取，用于每步填充并应用 token bitmask。
 212:             except Exception as e:
```

**读这段时抓住：**

- 单机时 ready set 可以直接使用；分布式时 ready 取交集，failed 取并集，偏保守保证各 rank 一致。
- 编译超时后会 cancel Future，并把失败对象写入 grammar backend cache。
- 这段解释了为什么 structured output 请求有时会先等待 grammar preprocessing，而不是立即进入 scheduler。


## XGrammar backend：token bitmask 路径

`XGrammarGrammarBackend` 使用 tokenizer 信息创建 `GrammarCompiler`，dispatch JSON/regex/EBNF 得到 compiled grammar。`XGrammarGrammar` 持有 `GrammarMatcher`，每步填 bitmask，然后在 GPU logits 上应用 bitmask。


In [ ]:
show_lines("python/sglang/srt/constrained/xgrammar_backend.py", 59, 130)
show_lines("python/sglang/srt/constrained/xgrammar_backend.py", 188, 245)


### `XGrammarGrammar`：matcher 状态如何变成 token bitmask

```python
# python/sglang/srt/constrained/xgrammar_backend.py:59-125
  59: class XGrammarGrammar(BaseGrammarObject):
  60: 
  61:     def __init__(
  62:         self,
  63:         matcher: GrammarMatcher,
  64:         vocab_size: int,
  65:         ctx: CompiledGrammar,
  66:         override_stop_tokens: Optional[Union[List[int], int]],
  67:         key_string: Optional[str] = None,
  68:         grammar_stats: Optional[GrammarStats] = GrammarStats(),
  69:     ) -> None:
  70:         super().__init__()
  71:         self.matcher = matcher
      # 注：每个请求独享的 xgrammar matcher，记录当前 FSM 状态并生成下一 token mask。
  72:         self.vocab_size = vocab_size
      # 注：grammar bitmask 的词表宽度必须与模型 vocab size 一致。
  73:         self.ctx = ctx
  74:         self.override_stop_tokens = override_stop_tokens
      # 注：xgrammar 使用模型 EOS/自定义 stop tokens，避免 grammar 终止语义和模型终止语义不一致。
  75:         self.accepted_tokens = []
      # 注：调试和 rollback 用的已接受 token 序列，matcher 拒绝 token 时用于错误上下文。
  76:         self.key_string = key_string
  77:         self.grammar_stats = grammar_stats
  78: 
  79:     def accept_token(self, token: int):
  80:         if not self.is_terminated():
  81:             self.current_token = token
  82:             accepted = self.matcher.accept_token(token)
      # 注：xgrammar matcher 是否接受刚采样出的 token；False 表示约束状态和生成 token 不一致。
  83:             if not accepted:
  84:                 # log for debugging
  85:                 raise ValueError(
  86:                     f"Tokens not accepted: {token}\n"
  87:                     f"Accepted tokens: {self.accepted_tokens}\n"
  88:                     f"Key string: {self.key_string}"
  89:                 )
  90:             else:
  91:                 self.accepted_tokens.append(token)
  92: 
  93:     def rollback(self, k: int):
  94:         self.matcher.rollback(k)
      # 注：关键调用：`self.matcher.rollback` - 撤回 matcher 的最近 k 个 token，供 speculative reject 或 jump-forward retokenize 后重放。
  95:         self.accepted_tokens = self.accepted_tokens[:-k]
      # 注：调试和 rollback 用的已接受 token 序列，matcher 拒绝 token 时用于错误上下文。
  96: 
  97:     def is_terminated(self):
  98:         return self.matcher.is_terminated()
      # 注：关键调用：`self.matcher.is_terminated` - 查询 grammar 是否已到终止状态，终止后不再推进 token mask。
  99: 
 100:     def allocate_vocab_mask(
 101:         self, vocab_size: int, batch_size: int, device
 102:     ) -> torch.Tensor:
 103:         return allocate_token_bitmask(batch_size, vocab_size)
      # 注：关键调用：`allocate_token_bitmask` - 为 constrained decoding 分配 token 级合法性 bitmask。
 104: 
 105:     def fill_vocab_mask(self, vocab_mask: torch.Tensor, idx: int) -> None:
 106:         self.matcher.fill_next_token_bitmask(vocab_mask, idx)
      # 注：关键调用：`self.matcher.fill_next_token_bitmask` - 把当前 grammar 状态下的合法 token 写入 bitmask。
 107: 
 108:     @staticmethod
 109:     def move_vocab_mask(vocab_mask: torch.Tensor, device) -> torch.Tensor:
 110:         return vocab_mask.to(device, non_blocking=True)
      # 注：关键调用：`vocab_mask.to` - 把 CPU 侧 token bitmask 异步搬到 logits 所在设备，采样前 mask kernel 才能直接读取。
 111: 
 112:     def apply_vocab_mask(self, logits: torch.Tensor, vocab_mask: torch.Tensor) -> None:
 113:         if logits.device.type in {"cuda", "xpu", "musa"}:
      # 注：GPU/XPU/MUSA logits 走设备侧 bitmask kernel，避免把 logits 拉回 CPU。
 114:             if _is_hip:
      # 注：ROCm/HIP 环境使用 CUDA 兼容实现；非 HIP GPU 默认走 Triton bitmask kernel。
 115:                 apply_token_bitmask_inplace_cuda(logits, vocab_mask)
 116:             else:
 117:                 apply_token_bitmask_inplace_triton(logits, vocab_mask)
      # 注：关键调用：`apply_token_bitmask_inplace_triton` - 在 GPU logits 上原地应用 token mask，非法 token 会被屏蔽。
 118:         elif logits.device.type == "npu":
      # 注：NPU 后端使用 sgl-kernel-npu 注册的 token bitmask op。
 119:             import sgl_kernel_npu  # noqa: F401
 120: 
 121:             torch.ops.npu.apply_token_bitmask(logits, vocab_mask)
 122:         else:
 123:             raise RuntimeError(f"Unsupported device: {logits.device.type}")
 124: 
 125:     def copy(self):
```

**读这段时抓住：**

- `GrammarMatcher.accept_token` 是 xgrammar 的状态推进；SGLang 额外记录 accepted tokens 方便报错和 rollback。
- `fill_next_token_bitmask` 把下一步合法 token 写入 bitmask；后续 logits kernel 只看这个 bitmask。
- `apply_vocab_mask` 按设备选择 Triton/CUDA/NPU kernel，因此 CPU-only 环境适合读/编译，不适合执行 mask kernel。


### `XGrammarGrammarBackend`：tokenizer 信息决定 grammar 能否编译

```python
# python/sglang/srt/constrained/xgrammar_backend.py:188-245
 188: class XGrammarGrammarBackend(BaseGrammarBackend):
 189:     def __init__(
 190:         self,
 191:         tokenizer,
 192:         vocab_size: int,
 193:         model_eos_token_ids: Optional[List[int]] = None,
 194:         any_whitespace: bool = True,
 195:     ):
 196:         super().__init__()
 197: 
 198:         if hasattr(tokenizer, "init_xgrammar"):
      # 注：自定义 tokenizer 可以提供专门的 xgrammar 初始化，绕开 Hugging Face TokenizerInfo 转换。
 199:             # For special tokenizer
 200:             tokenizer_info, override_stop_tokens = tokenizer.init_xgrammar()
 201: 
 202:             if tokenizer_info is None:
      # 注：自定义 tokenizer 明确表示不支持 xgrammar 时，backend 初始化失败并回退到错误路径。
 203:                 # Not supported tokenizer
 204:                 raise TokenizerNotSupportedError(
 205:                     f"Tokenizer type {type(tokenizer).__name__} is not supported by XGrammar"
 206:                 )
 207:         else:
 208:             # Create TokenizerInfo with model's EOS tokens as the authoritative stop tokens
 209:             # This ensures consistency between what the model considers EOS and what XGrammar uses
 210:             try:
 211:                 tokenizer_info = TokenizerInfo.from_huggingface(
      # 注：`tokenizer_info` 接收 `TokenizerInfo.from_huggingface` 的结果：把 Hugging Face tokenizer 转成 xgrammar 可用的 token 信息，并显式使用模型 EOS 作为 stop token。
 212:                     tokenizer, vocab_size=vocab_size, stop_token_ids=model_eos_token_ids
 213:                 )
 214:                 override_stop_tokens = None
 215:             except Exception as e:
 216:                 raise TokenizerNotSupportedError(
 217:                     f"Failed to create XGrammar TokenizerInfo from tokenizer: {e}"
 218:                 )
 219: 
 220:         self.grammar_compiler = GrammarCompiler(tokenizer_info=tokenizer_info)
      # 注：xgrammar compiler 持有 tokenizer 边界信息，后续把 JSON/regex/EBNF 编译成 matcher 可执行对象。
 221:         self.vocab_size = vocab_size
      # 注：grammar bitmask 的词表宽度必须与模型 vocab size 一致。
 222:         self.override_stop_tokens = override_stop_tokens
      # 注：xgrammar 使用模型 EOS/自定义 stop tokens，避免 grammar 终止语义和模型终止语义不一致。
 223:         self.any_whitespace = any_whitespace
      # 注：xgrammar 编译 schema 时的 whitespace 策略，影响 JSON 等格式中空白 token 的可接受性。
 224: 
 225:     @property
 226:     def is_support_token_filter(self):
 227:         return True
 228: 
 229:     @staticmethod
 230:     def allocate_vocab_mask(vocab_size: int, batch_size: int, device) -> torch.Tensor:
 231:         return allocate_token_bitmask(batch_size, vocab_size)
      # 注：关键调用：`allocate_token_bitmask` - 为 constrained decoding 分配 token 级合法性 bitmask。
 232: 
 233:     @staticmethod
 234:     def move_vocab_mask(vocab_mask: torch.Tensor, device) -> torch.Tensor:
 235:         return vocab_mask.to(device, non_blocking=True)
      # 注：关键调用：`vocab_mask.to` - 把 CPU 侧 token bitmask 异步搬到 logits 所在设备，采样前 mask kernel 才能直接读取。
 236: 
 237:     @staticmethod
 238:     def apply_vocab_mask(logits: torch.Tensor, vocab_mask: torch.Tensor) -> None:
 239:         if logits.device.type in {"cuda", "npu", "xpu", "musa"}:
 240:             if _is_hip:
      # 注：ROCm/HIP 环境使用 CUDA 兼容实现；非 HIP GPU 默认走 Triton bitmask kernel。
 241:                 apply_token_bitmask_inplace_cuda(logits, vocab_mask)
 242:             else:
 243:                 apply_token_bitmask_inplace_triton(logits, vocab_mask)
      # 注：关键调用：`apply_token_bitmask_inplace_triton` - 在 GPU logits 上原地应用 token mask，非法 token 会被屏蔽。
 244:         else:
 245:             raise RuntimeError(f"Unsupported device: {logits.device.type}")
```

**读这段时抓住：**

- backend 初始化时先把 Hugging Face tokenizer 转成 `TokenizerInfo`，失败则可能降级为 grammar_backend=none。
- `model_eos_token_ids` 被传入 tokenizer info，确保 grammar stop token 与模型 EOS 语义一致。
- `is_support_token_filter` 返回 True，strict thinking 等功能会检查这个能力。


## Compressed FSM / Jump-forward 是什么

在 regex/FSM 约束里，某些状态后面只有一条确定路径。例如 schema 中固定的字段名、标点、引号。逐 token 生成这些确定片段会浪费 decode 步。

SGLang 的 jump-forward 思路是：如果当前 FSM 状态能确定接下来的一段字符串，就直接生成这段字符串，再重新 tokenization / 更新 grammar 状态。`outlines_jump_forward.py` 中的注释直接写着 “Faster constrained decoding with jump forward decoding / compressed finite state machine.”


In [ ]:
show_lines("python/sglang/srt/constrained/outlines_jump_forward.py", 1, 40)
show_lines("python/sglang/srt/constrained/outlines_jump_forward.py", 144, 176)


### `OutlinesJumpForwardMap`：把确定性 FSM 边压缩成可跳过片段

```python
# python/sglang/srt/constrained/outlines_jump_forward.py:62-110
  62: def init_state_to_jump_forward(regex_string):
  63:     try:
  64:         regex_pattern = interegular.parse_pattern(regex_string)
  65:     except InvalidSyntax as e:
  66:         logger.warning(f"skip invalid regex: {regex_string}, {e=}")
  67:         return
  68: 
  69:     byte_fsm = make_byte_level_fsm(regex_pattern.to_fsm().reduce(), keep_utf8=True)
  70:     regex_fsm, _ = make_deterministic_fsm(byte_fsm)
  71: 
  72:     fsm_info: FSMInfo = regex_fsm.fsm_info
  73: 
  74:     symbol_to_id = fsm_info.alphabet_symbol_mapping
  75:     id_to_symbol = {}
  76:     for symbol, id_ in symbol_to_id.items():
  77:         id_to_symbol.setdefault(id_, []).append(symbol)
  78: 
  79:     transitions = fsm_info.transitions
  80: 
  81:     outgoings_ct = defaultdict(int)
  82:     # NOTE(lsyin): Final states can lead to terminate, so they have one outgoing edge naturally
  83:     for s in fsm_info.finals:
  84:         outgoings_ct[s] = 1
  85: 
  86:     state_to_jump_forward = {}
  87:     for (state, id_), next_state in transitions.items():
  88:         if id_ == fsm_info.alphabet_anything_value:
  89:             # Arbitrarily symbol cannot be recognized as jump forward
  90:             continue
  91: 
  92:         symbols = id_to_symbol[id_]
  93:         for c in symbols:
  94:             if len(c) > 1:
  95:                 # Skip byte level transitions like c = "5E"
  96:                 continue
  97: 
  98:             outgoings_ct[state] += 1
  99:             if outgoings_ct[state] > 1:
 100:                 if state in state_to_jump_forward:
 101:                     del state_to_jump_forward[state]
 102:                 break
 103: 
 104:             state_to_jump_forward[state] = JumpEdge(
 105:                 symbol=c,
 106:                 symbol_next_state=next_state,
 107:             )
 108: 
 109:     # Process the byte level jump forward
 110:     outgoings_ct = defaultdict(int)
```

**读这段时抓住：**

- `init_state_to_jump_forward` 遍历 FSM 边，只保留那些从某状态出发没有歧义的确定边。
- 一旦发现同一 state 有多个可行 symbol/byte，就删除 jump-forward 记录，避免错误跳过采样。
- 这就是 compressed FSM 的直觉：把线性确定路径压缩成一次字符串推进。


## 小测试：检查当前环境可用的 grammar 后端

这个测试不启动模型。它检查依赖是否可 import，并定位 SGLang 注册逻辑。Mac 上如果没有 GPU，`apply_vocab_mask` 的 CUDA/Triton 路径不应执行，但编译侧依赖仍可检查。


In [ ]:
for mod in ["xgrammar", "outlines", "llguidance"]:
    try:
        __import__(mod)
        print(mod, "available")
    except Exception as e:
        print(mod, "not available:", type(e).__name__, str(e)[:120])

show_lines("python/sglang/srt/constrained/base_grammar_backend.py", 205, 270)


## 贡献者注意点

- Grammar object 必须可复制，因为缓存里保存的是“初始 grammar”，每个请求要拿自己的状态副本。
- Jump-forward 需要 retokenize，tokenizer 的 byte/string 边界非常容易出错。
- 多 rank 下 grammar ready/failed 的同步必须保守，否则不同 rank 会对同一 batch 使用不同约束。
- Reasoning model 的 strict thinking 会复用 grammar/token filter 机制，不要把 structured output 简化成“JSON only”。
